# Direct Postgres Connection + Spark SQL

This notebook connects to `hive-postgres` using **psycopg2** (native Python, no JVM issues),  
loads the Hive Metastore tables into Pandas, then registers them as **Spark SQL temp views**  
using the **existing** Spark session so you can query with `spark.sql()`.

| Setting | Value |
|---|---|
| Container | `hive-postgres` |
| Database | `metastore` |
| User | `hive` |
| Password | `hive` |

## Step 1 — Install psycopg2 & Resolve Hostname

In [4]:
import subprocess, sys, socket

subprocess.run([sys.executable, '-m', 'pip', 'install', 'psycopg2-binary', '--quiet'], check=True)

POSTGRES_HOST = "hive-postgres"
try:
    resolved_ip = socket.gethostbyname(POSTGRES_HOST)
    print(f"Resolved '{POSTGRES_HOST}' -> {resolved_ip}")
except socket.gaierror as e:
    raise RuntimeError(
        f"Cannot resolve '{POSTGRES_HOST}': {e}\n"
        "Make sure the hive-postgres container is running."
    )

Resolved 'hive-postgres' -> 172.19.0.6


## Step 2 — Connect to Postgres & Load Tables into Pandas

In [5]:
import psycopg2
import psycopg2.extras
import pandas as pd

conn = psycopg2.connect(
    host=POSTGRES_HOST, port=5432,
    dbname='metastore', user='hive', password='hive'
)
print(f"Connected to {POSTGRES_HOST}:5432/metastore")

def pg_query(sql: str) -> pd.DataFrame:
    """Run a SQL query and return a Pandas DataFrame."""
    with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
        cur.execute(sql)
        return pd.DataFrame(cur.fetchall())

# Load the core Hive Metastore tables into Pandas
raw = {
    "dbs":          pg_query('SELECT * FROM "DBS"'),
    "tbls":         pg_query('SELECT * FROM "TBLS"'),
    "columns":      pg_query('SELECT * FROM "COLUMNS_V2"'),
    "sds":          pg_query('SELECT * FROM "SDS"'),
    "table_params": pg_query('SELECT * FROM "TABLE_PARAMS"'),
}

for name, df in raw.items():
    print(f"  {name:15s} — {len(df)} rows, {list(df.columns)}")

Connected to hive-postgres:5432/metastore
  dbs             — 1 rows, ['DB_ID', 'DESC', 'DB_LOCATION_URI', 'NAME', 'OWNER_NAME', 'OWNER_TYPE', 'CTLG_NAME']
  tbls            — 5 rows, ['TBL_ID', 'CREATE_TIME', 'DB_ID', 'LAST_ACCESS_TIME', 'OWNER', 'OWNER_TYPE', 'RETENTION', 'SD_ID', 'TBL_NAME', 'TBL_TYPE', 'VIEW_EXPANDED_TEXT', 'VIEW_ORIGINAL_TEXT', 'IS_REWRITE_ENABLED']
  columns         — 25 rows, ['CD_ID', 'COMMENT', 'COLUMN_NAME', 'TYPE_NAME', 'INTEGER_IDX']
  sds             — 5 rows, ['SD_ID', 'INPUT_FORMAT', 'IS_COMPRESSED', 'LOCATION', 'NUM_BUCKETS', 'OUTPUT_FORMAT', 'SERDE_ID', 'CD_ID', 'IS_STOREDASSUBDIRECTORIES']
  table_params    — 35 rows, ['TBL_ID', 'PARAM_KEY', 'PARAM_VALUE']


## Step 3 — Register as Spark SQL Temp Views

We push the Pandas DataFrames into Spark (using the **existing** active session — no new session created, no JVM conflict).

In [6]:
from pyspark.sql import SparkSession

# Reuse the active Spark session — do NOT create a new one.
# Creating a new session with different configs causes 'JavaPackage' errors.
spark = SparkSession.getActiveSession()

if spark is None:
    raise RuntimeError(
        "No active Spark session found.\n"
        "Run Business_Insight_Analysis.ipynb first (or any notebook that starts Spark), "
        "then come back and run this cell."
    )

print(f"Using existing Spark {spark.version} session.")

# Create temp views from the Pandas DataFrames loaded in Step 2
for view_name, pdf in raw.items():
    # Normalise column names to lowercase so Spark SQL works without quoting
    pdf.columns = [c.lower() for c in pdf.columns]
    spark.createDataFrame(pdf).createOrReplaceTempView(view_name)
    print(f"  Registered temp view: {view_name}")

print("\nAll views ready. Run spark.sql() in the cells below.")

RuntimeError: No active Spark session found.
Run Business_Insight_Analysis.ipynb first (or any notebook that starts Spark), then come back and run this cell.

## Query 1 — List All Hive Databases

In [7]:
spark.sql("""
    SELECT db_id, name AS database_name, desc AS description, owner_name
    FROM dbs
    ORDER BY db_id
""").show(truncate=False)

AttributeError: 'NoneType' object has no attribute 'sql'

## Query 2 — List All Hive Tables

In [ ]:
spark.sql("""
    SELECT d.name AS database_name, t.tbl_name AS table_name,
           t.owner, t.tbl_type AS table_type
    FROM tbls t
    JOIN dbs  d ON t.db_id = d.db_id
    ORDER BY d.name, t.tbl_name
""").show(truncate=False)

## Query 3 — Columns per Table

In [ ]:
spark.sql("""
    SELECT d.name AS database_name, t.tbl_name AS table_name,
           c.column_name, c.type_name AS data_type, c.integer_idx AS col_order
    FROM columns c
    JOIN sds  s ON c.cd_id = s.cd_id
    JOIN tbls t ON s.sd_id = t.sd_id
    JOIN dbs  d ON t.db_id = d.db_id
    ORDER BY d.name, t.tbl_name, c.integer_idx
""").show(50, truncate=False)

## Query 4 — HDFS Storage Locations

In [ ]:
spark.sql("""
    SELECT d.name AS database_name, t.tbl_name AS table_name,
           s.location AS hdfs_location, s.input_format
    FROM sds  s
    JOIN tbls t ON s.sd_id = t.sd_id
    JOIN dbs  d ON t.db_id = d.db_id
    ORDER BY d.name, t.tbl_name
""").show(truncate=False)

## Query 5 — Table Properties

In [ ]:
spark.sql("""
    SELECT t.tbl_name AS table_name, p.param_key, p.param_value
    FROM table_params p
    JOIN tbls         t ON p.tbl_id = t.tbl_id
    ORDER BY t.tbl_name, p.param_key
""").show(50, truncate=False)

## Custom SQL Query

Write any `spark.sql()` below using the views: `dbs`, `tbls`, `columns`, `sds`, `table_params`.

In [ ]:
spark.sql("""
    SELECT d.name AS database_name, COUNT(*) AS table_count
    FROM tbls t
    JOIN dbs  d ON t.db_id = d.db_id
    GROUP BY d.name
    ORDER BY table_count DESC
""").show()

## Close Postgres Connection

In [ ]:
conn.close()
print("Postgres connection closed.")